A test to see which number is more suitable for set_num_threads

In [ ]:
# import torch
# import time
#
# def benchmark_threads(num_threads, matrix_size=4000, iterations=20):
#     # Set the number of threads for this test run
#     torch.set_num_threads(num_threads)
#
#     # Create two massive random matrices to simulate deep learning workloads
#     A = torch.randn(matrix_size, matrix_size)
#     B = torch.randn(matrix_size, matrix_size)
#
#     # Warmup round (gets the CPU caches loaded and ready)
#     _ = torch.matmul(A, B)
#
#     # Start the actual timer
#     start_time = time.time()
#
#     # Run the heavy math multiple times
#     for _ in range(iterations):
#         _ = torch.matmul(A, B)
#
#     end_time = time.time()
#
#     print(f"Total time with {num_threads} threads: {end_time - start_time:.4f} seconds")
#
# print("--- PyTorch CPU Thread Benchmark ---")
# # Test with 4 threads
# benchmark_threads(4)
#
# # Test with 8 threads
# benchmark_threads(8)

8 threads is faster than 4 threads, so we will use 8 threads for the rest of the code

Cell 1: Imports and Device Setup

In [ ]:
import numpy as np
from ofdm.metrics import calculate_papr, calculate_cm
from ofdm.plots import plot_ccdf, plot_ccdf_compare
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, Subset
# from torchmin import Minimizer
import os
import time
import matplotlib.pyplot as plt
from IPython.display import clear_output, display

In [ ]:
samples_per_L = 10000

# Modulation scheme
mod = "16qam"
# mod = "qpsk"
print(f"Using {mod.upper()} modulation technique")

# Hyperparameters
epochs = 100
# using 0.001 for Adam and 0.01 for LBFGS as default values
lr = 0.001
# lr = 0.01
lr_list = str(float(lr)).split(".")
lr_str = f"dot{lr_list[1]}" if lr < 1 else f"{lr_list[0]}dot{lr_list[1]}"
print(f"Hyperparameters: epochs = {epochs}, learning rate = {lr} ({lr_str} used in naming files), batch size = None (for now)")

# Optimizer
opt = "Adam"
# opt = "LBFGS"
print(f"Using {opt} optimizer")

# Data splitting
train_size = 80
test_size = (100 - train_size) or 1
train_test_str = f"{train_size}_{test_size}"
if train_size < 100:
    one_batch = None
elif train_size == 100:
    one_batch = "00"
print(f"dataset is split into {train_size} training files and {test_size} batches ({train_test_str} used in naming files, while {one_batch} is the index of used batch if testing on 1 batch)")

if train_size == 100:
    print("This 1 batch is of index 00, and was already used in training")

batch_suffix = f"(Batch #{one_batch})" if one_batch else ""
params = f"{opt} optimizer {mod.upper()} lr = {lr} ({train_size} Training/{test_size if not one_batch else 1} Testing) {batch_suffix}"
print(params)

og_pt_dir = "./pt_dir/"
pt_dir = "./pt_dir_globalnorm/"
model_dir = "./new architecture/"
graph_dir = "./new architecture/"
os.makedirs(og_pt_dir, exist_ok=True)
os.makedirs(pt_dir, exist_ok=True)
os.makedirs(model_dir, exist_ok=True)
os.makedirs(graph_dir, exist_ok=True)
print(og_pt_dir, "is the location of the original pt files before implementing the brand new normalization method suggested by gemini")
print(f'"{pt_dir}", "{model_dir}", and "{graph_dir}" are the 3 locations for pt files, nn model weights and graphs respectively')

In [ ]:
# Explicitly tell PyTorch to utilize your 8 CPU cores for matrix math
# Check for GPU availability to drastically speed up training
# ! cuda is available only on nvidia gpu
torch.set_num_threads(8)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"PyTorch using {torch.get_num_threads()} threads on {device}")

In [ ]:
def denormalize(pred_norm, minmax):
    raw_min, raw_max = minmax
    pred_raw = ((pred_norm + 1.0) / 2.0) * (raw_max - raw_min) + raw_min if raw_max != raw_min else pred_norm
    return pred_raw

Cell 2: Custom Activation Function

In [ ]:
class TriangularActivation(nn.Module):
    """
    Implements the triangular activation function used in the paper's hidden layers.
    Mathematically equivalent to MATLAB's 'tribas': f(x) = max(1 - |x|, 0)
    """
    def forward(self, x):
        return torch.clamp(1.0 - torch.abs(x), min=0.0)

Cell 3: Highly Optimized Dataset Class

This is fast as it works with .pt directly without conversion

In [ ]:
class FastOFDMDataset(Dataset):
    def __init__(self, pt_folder_path, part):
        self.folder_path = pt_folder_path
        self.part = part

    def __len__(self):
        return 100 # Assuming exactly 100 pre-computed .pt files, regardless of train/test split

    def __getitem__(self, idx):
        # Instantly loads the pre-normalized tensors directly into memory!
        file_path = os.path.join(self.folder_path, f"{mod}_tx_rx_32_part_{idx:02d}_{self.part}.pt")
        # Load the dictionary
        data_pkg = torch.load(file_path, weights_only=False)

        # Return the tensors AND the scaling limits
        return (data_pkg['X_norm'], data_pkg['Y_norm'],
                data_pkg['X_min'], data_pkg['X_max'],
                data_pkg['Y_min'], data_pkg['Y_max'])

Cell 4: Neural Network Architecture

In [ ]:
class NNICFMapper(nn.Module):
    def __init__(self):
        super(NNICFMapper, self).__init__()
        # First hidden layer: 2 neurons
        self.hidden1 = nn.Linear(1, 2)
        # Second hidden layer: 1 neuron
        self.hidden2 = nn.Linear(2, 1)
        # Output layer: Maps back to the original subcarrier points
        self.output = nn.Linear(1, 1)
        # Custom triangular activation
        self.activation = TriangularActivation()

    def forward(self, x):
        x = self.activation(self.hidden1(x))
        x = self.activation(self.hidden2(x))
        x = self.output(x) # Standard linear output
        return x

Cell 5: Initialization and DataLoaders

In [ ]:
# Configuration

# Initialize DataLoaders with asynchronous loading (num_workers) and rapid memory transfer
train_dataset_real = FastOFDMDataset(pt_dir, part='real')
if train_size < 100:
    # ! note that the output of the Subset is saved in the original variable, may be important later
    train_dataset_real = Subset(train_dataset_real, range(train_size))  # Use the first train_size files for training

# ? should i change batch_size in the new architecture
train_loader_real = DataLoader(
    train_dataset_real,
    batch_size=None, # batch_size is None because the dataset returns a full batch of 10,000
    shuffle=True,
    # ! does not work locally on windows when changed to 2-4 for some reason
    num_workers=0,   # Adjust between 2-4 based on your CPU cores
    pin_memory=torch.cuda.is_available()
)

train_dataset_imag = FastOFDMDataset(pt_dir, part='imag')
if train_size < 100:
    train_dataset_imag = Subset(train_dataset_imag, range(train_size))  # Use the first train_size files for training

train_loader_imag = DataLoader(
    train_dataset_imag,
    batch_size=None,
    shuffle=True,
    num_workers=0,
    pin_memory=torch.cuda.is_available()
)

# Initialize models and send them to the GPU/device
Mod_Re_NN = NNICFMapper().to(device)
Mod_Im_NN = NNICFMapper().to(device)

# todo: see how can i use loss function to reduce ber
# Standard Mean Squared Error loss and optimizer
criterion = nn.MSELoss()
if opt == "Adam":
    optimizer_real = optim.Adam(Mod_Re_NN.parameters(), lr=lr)
    optimizer_imag = optim.Adam(Mod_Im_NN.parameters(), lr=lr)
elif opt == "LBFGS":
    optimizer_real = optim.LBFGS(Mod_Re_NN.parameters(), lr=lr, tolerance_change=1e-5, max_iter=5)
    optimizer_imag = optim.LBFGS(Mod_Im_NN.parameters(), lr=lr, tolerance_change=1e-5, max_iter=5)
# elif opt == "LM":
#     optimizer_real = Minimizer(Mod_Re_NN.parameters(), method='levenberg-marquardt')
#     optimizer_imag = Minimizer(Mod_Im_NN.parameters(), method='levenberg-marquardt')

Not using .compile() is actually better, it also avoids the hassle of cl.exe and needing to run "x64 Native Tools Command Prompt for VS".

dont know if thats the case with gpu or not tho

Cell 6: Training Loop (Real Module)

In [ ]:
# * in new architectur, takes ~30s per epoch for Adam optimizer, and ~60s per epoch for LGBFGS optimizer
# * in old architecture, takes ~10s per epoch for Adam optimizer
print("--- Starting Training for the Real NN Module ---")
loss_history = []  # Track losses for graphing
log_lines = []
total_time = 0

# ! if using .compile(), it is expected for the first epoch to take longer
for epoch in range(1, epochs + 1):
    start = time.time()
    epoch_loss = 0.0
    Mod_Re_NN.train()

    for X_batch, Y_batch, _, _, _, _ in train_loader_real:
        # Transfer data to GPU if possible
        X_batch, Y_batch = X_batch.to(device, non_blocking=True), Y_batch.to(device, non_blocking=True)

        # MEMORYLESS RESHAPE: Flatten into a 1D column
        X_batch_flat = X_batch.view(-1, 1)
        Y_batch_flat = Y_batch.view(-1, 1)

        if opt == "Adam":
            optimizer_real.zero_grad(set_to_none=True)
            # Predict on the flattened column
            predictions = Mod_Re_NN(X_batch_flat)
            loss = criterion(predictions, Y_batch_flat)
            loss.backward()
            optimizer_real.step()
            epoch_loss += loss.item()
        # elif opt == "LBFGS" or opt == "LM:
        elif opt == "LBFGS":
            # L-BFGS and LM require a "closure" function to recalculate the loss, but for LM it is often required to return the unreduced residuals
            def closure():
                optimizer_real.zero_grad(set_to_none=True)
                predictions = Mod_Re_NN(X_batch_flat)

                # if opt == "LM":
                #     # LM mathematically requires the raw residuals (the difference before squaring)
                #     residuals = predictions - Y_batch_flat

                loss = criterion(predictions, Y_batch_flat)
                loss.backward()
                return loss
            # Pass the closure to the optimizer
            optimizer_real.step(closure)
            # To track the loss for your graphs, you have to call it once outside
            epoch_loss += closure().item()

    avg_loss = epoch_loss / len(train_loader_real)
    loss_history.append(avg_loss)  # Store loss for this epoch

    end = time.time()
    total_time += end - start
    line = f'Real Module | Epoch {epoch}/{epochs} | Loss: {avg_loss:.6f} | Time: {end - start:.2f} seconds'
    log_lines.append(line)

    if epoch % 5 == 0 or epoch == epochs:
        clear_output(wait=True)
        plt.figure(figsize=(10, 7))
        plt.plot(range(1, len(loss_history) + 1), loss_history, marker='o', linestyle='-', linewidth=2, markersize=4)
        plt.xlabel('Epoch', fontsize=12)
        plt.ylabel('Loss (MSE)', fontsize=12)
        plt.title(f'Real Module Training Loss over Epochs\n{params}', fontsize=14, fontweight='bold')
        plt.xlim([0, epochs + 1])
        margin = (max(loss_history) - min(loss_history)) * 0.1 if len(loss_history) > 1 else 0.1
        plt.ylim([min(loss_history) - margin, max(loss_history) + margin])
        plt.grid(True, alpha=0.3)
        plt.tight_layout()
        plt.show()
        print(*log_lines, sep='\n')
    else:
        print(line)


print(f'Real NN Module Training Complete in {int(total_time // 60)}:{total_time % 60:.1f} minutes!')

# Plot loss vs epoch
plt.figure(figsize=(10, 7))
plt.plot(range(1, epochs + 1), loss_history, marker='o', linestyle='-', linewidth=2, markersize=4)
plt.xlabel('Epoch', fontsize=12)
plt.ylabel('Loss (MSE)', fontsize=12)
plt.title(f'Real Module Training Loss over Epochs\n{params}', fontsize=14, fontweight='bold')
plt.xlim([0, epochs + 1])
margin = (max(loss_history) - min(loss_history)) * 0.1
plt.ylim([min(loss_history) - margin, max(loss_history) + margin])
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(graph_dir, f"{opt}_{mod}_real_loss_{train_test_str}_{lr_str}.png"), dpi=150)
plt.show()

Cell 7: Training Loop (Imaginary Module)

In [ ]:
print("--- Starting Training for the Imaginary NN Module ---")
loss_history = []  # Track losses for graphing
log_lines = []
total_time = 0

for epoch in range(1, epochs + 1):
    start = time.time()
    epoch_loss = 0.0
    Mod_Im_NN.train()

    for X_batch, Y_batch, _, _, _, _ in train_loader_imag:
        # Transfer data to GPU if possible
        X_batch, Y_batch = X_batch.to(device, non_blocking=True), Y_batch.to(device, non_blocking=True)

        # MEMORYLESS RESHAPE: Flatten into a 1D column
        X_batch_flat = X_batch.view(-1, 1)
        Y_batch_flat = Y_batch.view(-1, 1)

        if opt == "Adam":
            optimizer_imag.zero_grad(set_to_none=True)
            # Predict on the flattened column
            predictions = Mod_Im_NN(X_batch_flat)
            loss = criterion(predictions, Y_batch_flat)
            loss.backward()
            optimizer_imag.step()
            epoch_loss += loss.item()
        # elif opt == "LBFGS" or opt == "LM:
        elif opt == "LBFGS":
            # L-BFGS and LM require a "closure" function to recalculate the loss, but for LM it is often required to return the unreduced residuals
            def closure():
                optimizer_imag.zero_grad(set_to_none=True)
                predictions = Mod_Im_NN(X_batch_flat)

                # if opt == "LM":
                #     # LM mathematically requires the raw residuals (the difference before squaring)
                #     residuals = predictions - Y_batch_flat

                loss = criterion(predictions, Y_batch_flat)
                loss.backward()
                return loss
            # Pass the closure to the optimizer
            optimizer_imag.step(closure)
            # To track the loss for your graphs, you have to call it once outside
            epoch_loss += closure().item()

    avg_loss = epoch_loss / len(train_loader_imag)
    loss_history.append(avg_loss)  # Store loss for this epoch

    end = time.time()
    total_time += end - start
    line = f'Imaginary Module | Epoch {epoch}/{epochs} | Loss: {avg_loss:.6f} | Time: {end - start:.2f} seconds'
    log_lines.append(line)

    if epoch % 5 == 0 or epoch == epochs:
        clear_output(wait=True)
        plt.figure(figsize=(10, 7))
        plt.plot(range(1, len(loss_history) + 1), loss_history, marker='o', linestyle='-', linewidth=2, markersize=4)
        plt.xlabel('Epoch', fontsize=12)
        plt.ylabel('Loss (MSE)', fontsize=12)
        plt.title(f'Imaginary Module Training Loss over Epochs\n{params}', fontsize=14, fontweight='bold')
        plt.xlim([0, epochs + 1])
        margin = (max(loss_history) - min(loss_history)) * 0.1 if len(loss_history) > 1 else 0.1
        plt.ylim([min(loss_history) - margin, max(loss_history) + margin])
        plt.grid(True, alpha=0.3)
        plt.tight_layout()
        plt.show()
        print(*log_lines, sep='\n')
    else:
        print(line)


print(f'Imaginary NN Module Training Complete in {int(total_time // 60)}:{total_time % 60:.1f} minutes!')

# Plot loss vs epoch
plt.figure(figsize=(10, 7))
plt.plot(range(1, epochs + 1), loss_history, marker='o', linestyle='-', linewidth=2, markersize=4)
plt.xlabel('Epoch', fontsize=12)
plt.ylabel('Loss (MSE)', fontsize=12)
plt.title(f'Imaginary Module Training Loss over Epochs\n{params}', fontsize=14, fontweight='bold')
plt.xlim([0, epochs + 1])
margin = (max(loss_history) - min(loss_history)) * 0.1
plt.ylim([min(loss_history) - margin, max(loss_history) + margin])
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(graph_dir, f"{opt}_{mod}_imag_loss_{train_test_str}_{lr_str}.png"), dpi=150)
plt.show()


Cell 8: Saving the Models

In [ ]:
# Save the trained models
torch.save(Mod_Re_NN.state_dict(), os.path.join(model_dir, f"{opt}_{mod}_mod_re_weights_{train_test_str}_{lr_str}.pth"))
torch.save(Mod_Im_NN.state_dict(), os.path.join(model_dir, f"{opt}_{mod}_mod_im_weights_{train_test_str}_{lr_str}.pth"))
print("\nTraining complete and weights saved!")

Since we already saved the models, we can comment out most of the previous code blocks

Cell 9: Testing

In [ ]:
# 1. Load the Saved Models
Test_Mod_Re = NNICFMapper().to(device)
Test_Mod_Im = NNICFMapper().to(device)

Test_Mod_Re.load_state_dict(torch.load(os.path.join(model_dir, f"{opt}_{mod}_mod_re_weights_{train_test_str}_{lr_str}.pth"), weights_only=True))
Test_Mod_Im.load_state_dict(torch.load(os.path.join(model_dir, f"{opt}_{mod}_mod_im_weights_{train_test_str}_{lr_str}.pth"), weights_only=True))

Test_Mod_Re.eval()
Test_Mod_Im.eval()

In [ ]:
if one_batch:
    # 2. Grab one batch of data to test (Load the dictionary packages)
    pkg_real = torch.load(os.path.join(pt_dir, f"{mod}_tx_rx_32_part_{one_batch}_real.pt"), weights_only=False)
    pkg_imag = torch.load(os.path.join(pt_dir, f"{mod}_tx_rx_32_part_{one_batch}_imag.pt"), weights_only=False)
    # Extract tensors
    test_X_real, test_Y_real = pkg_real['X_norm'], pkg_real['Y_norm']
    test_X_imag, test_Y_imag = pkg_imag['X_norm'], pkg_imag['Y_norm']

    # 3. Generate Predictions (Reshape, Predict, Reshape back)
    with torch.no_grad():
        # Memoryless flattening
        X_real_flat = test_X_real.view(-1, 1).to(device)
        X_imag_flat = test_X_imag.view(-1, 1).to(device)

        predicted_real = Test_Mod_Re(X_real_flat).view_as(test_X_real).cpu().numpy()
        predicted_imag = Test_Mod_Im(X_imag_flat).view_as(test_X_imag).cpu().numpy()

    # Denormalize using exact dictionary limits
    pred_denorm_real = denormalize(predicted_real, (pkg_real['X_min'], pkg_real['X_max']))
    pred_denorm_imag = denormalize(predicted_imag, (pkg_imag['X_min'], pkg_imag['X_max']))

    # 4. Reconstruct the Complex OFDM Signals
    original_complex = test_X_real.numpy() + 1j * test_X_imag.numpy()
    clipped_complex = test_Y_real.numpy() + 1j * test_Y_imag.numpy()
    predicted_complex = pred_denorm_real + 1j * pred_denorm_imag
elif train_size < 100:
    # 2. Setup the Test Data (Files train_size-99)
    test_dataset_real = Subset(FastOFDMDataset(pt_dir, part='real'), range(train_size, 100))
    test_dataset_imag = Subset(FastOFDMDataset(pt_dir, part='imag'), range(train_size, 100))

    # We set shuffle=False to ensure real and imag batches stay perfectly aligned
    test_loader_real = DataLoader(test_dataset_real, batch_size=None, shuffle=False)
    test_loader_imag = DataLoader(test_dataset_imag, batch_size=None, shuffle=False)

    # 3. Generate Predictions for all test_size test files
    all_original, all_clipped, all_predicted = [], [], []

    test_loss_real = 0.0
    test_loss_imag = 0.0
    with torch.no_grad():
        for (X_real, Y_real, X_r_min, X_r_max, _, _), (X_imag, Y_imag, X_i_min, X_i_max, _, _) in zip(test_loader_real, test_loader_imag):
            # Move inputs to device
            X_real, X_imag = X_real.to(device), X_imag.to(device)

            # 1. Flatten for memoryless prediction
            X_real_flat = X_real.view(-1, 1)
            X_imag_flat = X_imag.view(-1, 1)

            # Predict
            # ! shouldnt use .numpy() method because of criterion
            pred_real_flat = Test_Mod_Re(X_real_flat)
            pred_imag_flat = Test_Mod_Im(X_imag_flat)

            # 2. Calculate MSE Loss on the flat tensors
            loss_real = criterion(pred_real_flat, Y_real.view(-1, 1))
            loss_imag = criterion(pred_imag_flat, Y_imag.view(-1, 1))

            test_loss_real += loss_real.item()
            test_loss_imag += loss_imag.item()

            # 3. Reshape back to the original array shape (e.g. [10000, 1024])
            pred_real = pred_real_flat.view_as(X_real).cpu().numpy()
            pred_imag = pred_imag_flat.view_as(X_imag).cpu().numpy()

            # 4. Denormalize using the exact physical limits from the dictionary!
            pred_denorm_real = denormalize(pred_real, (X_r_min, X_r_max))
            pred_denorm_imag = denormalize(pred_imag, (X_i_min, X_i_max))

            # 5. Reconstruct complex signals
            orig_complex = X_real.cpu().numpy() + 1j * X_imag.cpu().numpy()
            clip_complex = Y_real.cpu().numpy() + 1j * Y_imag.cpu().numpy()
            pred_complex = pred_denorm_real + 1j * pred_denorm_imag

            all_original.append(orig_complex)
            all_clipped.append(clip_complex)
            all_predicted.append(pred_complex)

    # Print the final average test metrics
    avg_test_loss_real = test_loss_real / len(test_loader_real)
    avg_test_loss_imag = test_loss_imag / len(test_loader_imag)

    print(f'\n--- Test Set Metrics ({test_size} Unseen Files) ---')
    print(f'Average Real Module MSE Loss: {avg_test_loss_real:.6f}')
    print(f'Average Imag Module MSE Loss: {avg_test_loss_imag:.6f}')

    # 4. Concatenate the test_size batches into one massive array for evaluation
    original_complex = np.concatenate(all_original, axis=0)
    clipped_complex = np.concatenate(all_clipped, axis=0)
    predicted_complex = np.concatenate(all_predicted, axis=0)

    print(f'Testing complete! Aggregated {test_size * samples_per_L} OFDM symbols.')

Cell 10: Plotting

In [ ]:
# 5. Calculate PAPR and CM for CCDF
orig_papr, clip_papr, pred_papr = [], [], []
orig_cm, clip_cm, pred_cm = [], [], []

# Iterate through the arrays
for i in range(test_size * samples_per_L if not one_batch else samples_per_L):
    orig_papr.append(calculate_papr(original_complex[i]))
    clip_papr.append(calculate_papr(clipped_complex[i]))
    pred_papr.append(calculate_papr(predicted_complex[i]))

    orig_cm.append(calculate_cm(original_complex[i]))
    clip_cm.append(calculate_cm(clipped_complex[i]))
    pred_cm.append(calculate_cm(predicted_complex[i]))

orig_papr, clip_papr, pred_papr = np.array(orig_papr), np.array(clip_papr), np.array(pred_papr)
orig_cm, clip_cm, pred_cm = np.array(orig_cm), np.array(clip_cm), np.array(pred_cm)

# 6. Plot the CCDF
title = f'NNICF Predicted OFDM\n{params}'
labels = ['Original OFDM', 'ICF', 'NNICF Predicted OFDM']
papr_list = [orig_papr, clip_papr, pred_papr]
cm_list = [orig_cm, clip_cm, pred_cm]

# plot_ccdf_compare(papr_list, f'Original vs ICF vs {title}', labels)
plot_ccdf_compare(cm_list, f'Original vs ICF vs {title}', labels, metric="CM")

# fixme: both percentile and max are not the most effecient solution

# todo: find a way to embed the floor part into the plotting function
# 1. Define the target y-levels (probabilities)
papr_target_y = 1e-4
cm_target_y = 1e-3

# 2. Convert CCDF y-level to a standard percentile (e.g., 1e-4 becomes 99.99)
papr_percentile = (1.0 - papr_target_y) * 100.0
cm_percentile = (1.0 - cm_target_y) * 100.0

# 3. Extract the exact x-axis values (PAPR/CM) where the line cuts the graph
# (This completely replaces the need for y_axis, np.where, and manual sorting!)
papr_vlines = [
    np.percentile(orig_papr, papr_percentile),
    np.percentile(clip_papr, papr_percentile),
    np.percentile(pred_papr, papr_percentile)
]

cm_vlines = [
    np.percentile(orig_cm, cm_percentile),
    np.percentile(clip_cm, cm_percentile),
    np.percentile(pred_cm, cm_percentile)
]

# cm_vlines = [
#     np.max(orig_cm),
#     np.max(clip_cm),
#     np.max(pred_cm)
# ]

# # 4. Plot!
# # papr_image_path = os.path.join(graph_dir, f"{opt}_{mod}_papr_{train_test_str}_{lr_str}.png")
# # plot_ccdf(pred_papr, title, metric="papr", vlines=papr_vlines, save=papr_image_path)
# plot_ccdf(pred_papr, title, metric="papr", vlines=papr_vlines)

cm_image_path = os.path.join(graph_dir, f"{opt}_{mod}_cm_{train_test_str}_{lr_str}.png")
plot_ccdf(pred_cm, title, metric="cm", vlines=cm_vlines, save=cm_image_path)